# Task 3 — U-Net Semantic Segmentation on Stanford Background Dataset

This notebook runs all three loss-function experiments and logs results to TensorBoard.

**Experiments**
| # | Loss | Config key |
|---|------|------------|
| 1 | Cross-Entropy only | `loss_type='ce'` |
| 2 | Dice Loss only | `loss_type='dice'` |
| 3 | CE + Dice (combined) | `loss_type='combined'` |

**Before running**: download the Stanford Background Dataset and place it at `data/stanford_background/` (relative to the repo root), with sub-folders `images/` and `labels/`.

In [ ]:
import sys, os
# Make sure the repo root is on the path so `src` is importable
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Imports

In [ ]:
from src.config  import Config
from src.dataset import get_dataloaders
from src.model   import UNet
from src.losses  import get_loss_fn
from src.trainer import Trainer
from src.utils   import load_checkpoint, visualize_predictions

## 2. Shared hyperparameters

Edit the cell below to change any parameter. The `loss_type` field is overridden per-experiment.

In [ ]:
BASE_CFG = dict(
    data_root      = "../data/stanford_background",
    num_classes    = 8,
    input_size     = (256, 256),
    val_split      = 0.2,
    random_seed    = 42,

    # ── Training ──────────────────────────────────────────────────────────
    batch_size     = 8,
    num_epochs     = 50,
    learning_rate  = 1e-3,
    weight_decay   = 1e-4,
    num_workers    = 4,

    # ── Model ─────────────────────────────────────────────────────────────
    base_channels  = 64,

    # ── Optimizer / Scheduler ─────────────────────────────────────────────
    optimizer      = "adam",
    scheduler      = "cosine",

    # ── Dice-specific (used when loss_type != 'ce') ────────────────────────
    dice_weight    = 0.5,
    dice_smooth    = 1.0,

    # ── Logging ───────────────────────────────────────────────────────────
    log_dir        = "../runs",
    checkpoint_dir = "../checkpoints",
    save_every     = 10,
)

## 3. Helper: run one experiment

In [ ]:
def run_experiment(loss_type: str, extra_cfg: dict = None) -> float:
    """
    Train U-Net with the given loss_type.
    Returns the best validation mIoU achieved.
    """
    cfg_kwargs = {**BASE_CFG, "loss_type": loss_type}
    if extra_cfg:
        cfg_kwargs.update(extra_cfg)

    cfg = Config(**cfg_kwargs)
    print(f"\n{'='*60}")
    print(f"Experiment: {cfg.experiment_name}")
    print(f"{'='*60}")

    train_loader, val_loader = get_dataloaders(cfg)

    model   = UNet(num_classes=cfg.num_classes, base_channels=cfg.base_channels)
    loss_fn = get_loss_fn(cfg)

    print(f"Model parameters: {model.count_parameters():,}")

    trainer = Trainer(model, loss_fn, train_loader, val_loader, cfg, device)
    best_miou = trainer.train()
    return best_miou

## 4. Experiment 1 — Cross-Entropy Loss

In [ ]:
miou_ce = run_experiment("ce")
print(f"\n[CE]  Best val mIoU = {miou_ce:.4f}")

## 5. Experiment 2 — Dice Loss

In [ ]:
miou_dice = run_experiment("dice")
print(f"\n[Dice]  Best val mIoU = {miou_dice:.4f}")

## 6. Experiment 3 — Combined Loss (CE + Dice)

In [ ]:
miou_combined = run_experiment("combined")
print(f"\n[Combined]  Best val mIoU = {miou_combined:.4f}")

## 7. Results summary

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Loss Function": ["Cross-Entropy", "Dice", "CE + Dice"],
    "Best val mIoU": [miou_ce, miou_dice, miou_combined],
})
results["Best val mIoU"] = results["Best val mIoU"].map("{:.4f}".format)
print(results.to_string(index=False))

## 8. Launch TensorBoard

Run the cell below to open TensorBoard inline (works in Jupyter / VS Code).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../runs

## 9. Visualise predictions from the best model

Change `BEST_CKPT` to whichever checkpoint you want to inspect.

In [ ]:
import matplotlib.pyplot as plt

BEST_LOSS_TYPE = "combined"   # change to "ce" or "dice" as needed

cfg = Config(**{**BASE_CFG, "loss_type": BEST_LOSS_TYPE})
_, val_loader = get_dataloaders(cfg)

model = UNet(num_classes=cfg.num_classes, base_channels=cfg.base_channels).to(device)
ckpt_path = f"../checkpoints/{cfg.experiment_name}/best.pth"
load_checkpoint(model, ckpt_path, device=device)
model.eval()

images, labels = next(iter(val_loader))
with torch.no_grad():
    logits = model(images.to(device))

fig = visualize_predictions(
    images, labels, logits,
    class_names=cfg.CLASS_NAMES,
    save_path="../outputs/predictions.png",
    n_samples=4,
)
plt.show()

## 10. Ablation: vary learning rate

Uncomment and run to sweep learning rates with the combined loss.

In [ ]:
# for lr in [1e-4, 5e-4, 1e-3, 3e-3]:
#     miou = run_experiment("combined", extra_cfg={"learning_rate": lr, "num_epochs": 30})
#     print(f"lr={lr}  →  best mIoU={miou:.4f}")